In [1]:
#import twisted.internet.asyncioreactor
#twisted.internet.asyncioreactor.install()


import pandas as pd
import requests

import os
from pathlib import Path
import sys

import asyncio

from datetime import date, datetime, timedelta
from typing import TypedDict, List

import boto3
import json

from kayak.utils import setup_logger


In [2]:
import sys
print(sys.executable)

c:\Users\pfelz\anaconda3\envs\jedha_dts_lead\python.exe


In [3]:
import kayak
print(kayak.__file__)

C:\Pieric\datascience\JEDHA\Data Science Fundamentals\PYTHON\workspace\projects\jedha-dts-data-collect-mgmt-kayak\src\kayak\__init__.py


### Setting secrets ###

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display

style = {'description_width': 'initial'}
input_layout = widgets.Layout(width='400px')
box_layout = widgets.Layout(width='420px', margin='0 auto')

access_key = widgets.Password(description="AWS Key:")
secret_key = widgets.Password(description="AWS Secret:")
weatherforecats_api_key = widgets.Password(description="OpenWeather map api key:")
rds_postgresql_kayak_uri = widgets.Password(description="Rds postgresql kayak db URI:")

button = widgets.Button(description="Set secrets")
output = widgets.Output()

def set_env(_):
    os.environ["AWS_ACCESS_KEY_ID"] = access_key.value
    os.environ["AWS_SECRET_ACCESS_KEY"] = secret_key.value
    os.environ["OPENWEATHER_API_KEY"] = weatherforecats_api_key.value
    os.environ["RDS_KAYAK_POSTGRESQL_URI"] = rds_postgresql_kayak_uri.value
    os.environ["AWS_DEFAULT_REGION"] = "eu-west-3"

    with output:
        output.clear_output()
        print("Env variables set")

button.on_click(set_env)

form = widgets.VBox(
    [
        access_key,
        secret_key,
        weatherforecats_api_key,
        rds_postgresql_kayak_uri,
        button,
        output
    ],
    layout=box_layout
)

display(form)


### Define Directories ###

In [4]:
from kayak.config.paths import DATA_DIR

#DATA_DIR = BASE_DIR / "data"
BACKUP_DIR = DATA_DIR / "backup"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"

def remove_file_if_exists(relative_path:str):
    if os.path.exists(relative_path):
          os.remove(relative_path)

def get_destinations_file_name() -> str:
     return "destinations.json"


### Lookup target destinations ###

In [5]:

DESTINATIONS_INPUT_FILE_PATH = INPUT_DIR / get_destinations_file_name()

with open(DESTINATIONS_INPUT_FILE_PATH, "r", encoding="utf-8") as f:
    destinations_input_info_list = json.load(f)
print('ok')

ok


In [ ]:
from kayak.config.paths import LOG_DIR
from kayak.utils import setup_logger

log_file = LOG_DIR / "kayak.log"

root_logger = setup_logger("app", log_file=str(log_file), console=False)

### Run Data ingestion Pipeline ###
##### (Collect Destination gps coordinates, collect weather forecast, calculate destination weather scores over the period and finally scarp hotels) #####

In [13]:
from kayak.pipelines import IngestionPipeline

ingestion_pipeline_logger = root_logger.getChild("ingestion-pipeline")

ingestion_pipeline = IngestionPipeline(destinations_input_info_list=destinations_input_info_list,output_dir=OUTPUT_DIR,logger=ingestion_pipeline_logger)
ingestion_outputs = ingestion_pipeline.run()
assert ingestion_outputs.destination_coordinates_output_filepath.exists()
assert ingestion_outputs.weather_forecast_output_filepath.exists()
assert ingestion_outputs.destination_weather_score_period_file_path.exists()
assert ingestion_outputs.hotel_search_param_file_name.exists()
assert ingestion_outputs.hotel_search_result_file_path.exists()

### Check data ###

##### Destination gps coordinates #####

In [14]:
dest_coordinates_dtf = pd.read_csv(str(ingestion_outputs.destination_coordinates_output_filepath), encoding='utf-8',sep=',')
dest_coordinates_dtf.head(2)

,destination_id,destination_key,destination,gps_long,gps_lat
0,1,Mont Saint Michel,Mont Saint-Michel,-1.507013,48.609526
1,2,Saint-Malo,Saint-Malo,-2.026041,48.649518


##### Weather forecasts #####

In [15]:
weather_forecast_dtf = pd.read_csv(str(ingestion_outputs.weather_forecast_output_filepath), encoding='utf-8',sep=',')
weather_forecast_dtf.head(2)

,destination_id,destination,dt,temp,temp_min,temp_max,pressure,humidity,weather_main,weather_descr,clouds_info_dict,pop
0,1,Mont Saint-Michel,2026-04-21 12:00:00+00:00,13.98,13.98,15.81,1019,54,Clouds,few clouds,{'all': 22},0.0
1,1,Mont Saint-Michel,2026-04-21 15:00:00+00:00,15.15,15.15,16.20,1018,56,Clouds,scattered clouds,{'all': 42},0.0


##### Destination weather scores over forecast period #####

In [16]:
destination_weather_scores_dtf = pd.read_csv(str(ingestion_outputs.destination_weather_score_period_file_path), encoding='utf-8',sep=',')
destination_weather_scores_dtf.sort_values(['weather_rank'], ascending=[True]).head(2)

,destination_id,destination,weather_median_score,avg_temp,weather_rank,start_date,end_date
21,22,Aix-en-Provence,10.0,17.71950,1,2026-04-21 12:00:00+00:00,2026-04-26 09:00:00+00:00
20,21,Marseille,10.0,17.68425,2,2026-04-21 12:00:00+00:00,2026-04-26 09:00:00+00:00


##### Hotel search common params #####

In [17]:
dtf = pd.read_csv(str(ingestion_outputs.hotel_search_param_file_name), encoding='utf-8',sep=',')
dtf.head(2)

,search_id,checkin_date,checkout_date,destination_country,destination_type,group_adults,group_children
0,1280a9ab-ebe0-4cdf-a340-74af2b3a02a2,2026-04-21,2026-04-25,France,city,2,0


##### Hotel sraping result #####

In [18]:
dtf = pd.read_csv(str(ingestion_outputs.hotel_search_result_file_path), encoding='utf-8',sep=',')
dtf.sort_values(['destination_id', 'score'], ascending = [True, False], inplace=True)
dtf.head(2)

,destination_id,hotel_name,url,gps_lat,gps_long,score,description,address,search_id
283,1,Chez Stéphanie et Matthias !,https://www.booking.com/hotel/fr/chez-stephani...,48.599615,-1.421002,10.0,"Situé à Servon, l’hébergement Chez Stéphanie e...","18 Rue du Pont Morin, 50170 Servon, France",1280a9ab-ebe0-4cdf-a340-74af2b3a02a2
291,1,Au Mont Chez Nous Jaccuzi et Sauna,https://www.booking.com/hotel/fr/26-route-de-l...,48.613455,-1.485872,9.5,"Proposant un salon commun, l’hébergement Au Mo...","26 Route de la Rive Ardevon, 50170 Pontorson, ...",1280a9ab-ebe0-4cdf-a340-74af2b3a02a2


#### Plot top destinations in a map ####

In [16]:
destination_weather_scores_dtf.dtypes

destination_id            int64
destination              object
weather_median_score    float64
avg_temp                float64
weather_rank              int64
start_date               object
end_date                 object
dtype: object

##### Preprocessing to display top destinations on a map  (join gps coordinates, typing...) #####

In [19]:
TOP_N_DESTINATION = 5

top_destination_dtf = destination_weather_scores_dtf.sort_values(['weather_rank'], ascending=[True]).head(TOP_N_DESTINATION)
top_destination_dtf = pd.merge(left=top_destination_dtf, right=dest_coordinates_dtf, on='destination_id', how='left')
top_destination_dtf.drop(['destination_y'], axis=1, inplace=True)
top_destination_dtf.rename(columns={'destination_x':'destination'}, inplace=True)
top_destination_dtf['gps_long']= top_destination_dtf['gps_long'].astype(float)
top_destination_dtf['gps_lat'] = top_destination_dtf['gps_lat'].astype(float)
top_destination_dtf['start_date'] = pd.to_datetime(top_destination_dtf['start_date'], utc=True)
top_destination_dtf['end_date'] = pd.to_datetime(top_destination_dtf['end_date'], utc=True)
top_destination_dtf.head(2)


,destination_id,destination,weather_median_score,avg_temp,weather_rank,start_date,end_date,destination_key,gps_long,gps_lat
0,22,Aix-en-Provence,10.0,17.71950,1,2026-04-21 12:00:00+00:00,2026-04-26 09:00:00+00:00,Aix en Provence,5.447474,43.529842
1,21,Marseille,10.0,17.68425,2,2026-04-21 12:00:00+00:00,2026-04-26 09:00:00+00:00,Marseille,5.369953,43.296174


In [13]:
import plotly.express as px
# play with the color: continous/numerical
#px.set_mapbox_access_token(open(".mapbox_token").read())
weather_forecast_start_date = top_destination_dtf.iloc[0]['start_date']
weather_forecast_end_date = top_destination_dtf.iloc[0]['end_date']
fig = px.scatter_mapbox(top_destination_dtf, lat="gps_lat", lon="gps_long", color="weather_rank", text='destination', 
#                         size="hour",
                        #color_continuous_scale=px.colors.cyclical.IceFire,   
                        color_continuous_scale="RdYlGn_r",                     
                        zoom=4.5,
                       width=900,
                        height=600,
                        labels={'median_temp' : 'median temperature'},
                        title='Top 5 destinations ' + weather_forecast_start_date.strftime("%d-%m-%Y") + ' to ' + weather_forecast_end_date.strftime("%d-%m-%Y"))
fig.update_traces(marker=dict(size=20))
fig.update_layout(mapbox_style="open-street-map")
fig.show()

In [14]:
#!pip show kaleido
import importlib

print(importlib.util.find_spec("kaleido") is not None)

True


In [15]:
#fig.write_image("kayak_top5destinations.png")
fig.write_html("kayak_top5destinations.html")

#### Display Top-20 hotels for each of top-5 destinations in a map ####

##### Select Top-20 hotels for each of top-5 destinations #####

In [20]:
destination_hotels_dtf = pd.read_csv(str(ingestion_outputs.hotel_search_result_file_path), encoding='utf-8',sep=',')

top_hotels_dtf  = destination_hotels_dtf[destination_hotels_dtf['destination_id'].isin(top_destination_dtf['destination_id'].unique())].copy()
top_hotels_dtf.fillna({'score':0}, inplace=True)
top_hotels_dtf['hotel_rank'] = top_hotels_dtf.groupby('destination_id')['score'].rank(method='first',ascending=False).astype(int)
top_hotels_dtf  = top_hotels_dtf[(top_hotels_dtf['hotel_rank'] <= 20 )]

# result just to display the two best rated hotels of the two top destinations in term of weather rank
result = top_hotels_dtf[
    (top_hotels_dtf['destination_id'].isin(top_destination_dtf.loc[top_destination_dtf['weather_rank']<=2, 'destination_id'])) &
    (top_hotels_dtf['hotel_rank'] <= 2)
].sort_values(['destination_id', 'hotel_rank'])

#top_hotels_dtf.dtypes
result

,destination_id,hotel_name,url,gps_lat,gps_long,score,description,address,search_id,hotel_rank
831,21,Entre Plages et Jardins,https://www.booking.com/hotel/fr/entre-mer-et-...,43.250530,5.381482,9.5,"Offrant une vue sur le jardin, l’hébergement E...","72 Boulevard du Sablier, 13008 Marseille, France",1280a9ab-ebe0-4cdf-a340-74af2b3a02a2,1
477,21,Le Central Parc - T2-Spacieux-Clim-Wifi,https://www.booking.com/hotel/fr/le-central-pa...,43.281857,5.396548,9.4,L’hébergement 5 étoiles Le Central Parc - T2-S...,"27 Rue Paul Casimir, 13010 Marseille, France",1280a9ab-ebe0-4cdf-a340-74af2b3a02a2,2
821,22,T3 Centre ville Aix-en-Provence,https://www.booking.com/hotel/fr/t3-centre-vil...,43.521985,5.453696,9.8,L’hébergement T3 Centre ville Aix-en-Provence ...,"Bat D 1er etage 29 Avenue Saint-Jérôme, 13100 ...",1280a9ab-ebe0-4cdf-a340-74af2b3a02a2,1
807,22,Bed and Breakfast Jardin de Marie,https://www.booking.com/hotel/fr/le-jardin-de-...,43.524902,5.450541,9.1,"Situé à Aix-en-Provence, le Bed and Breakfast ...","47, rue Roux Alphéran, 13100 Aix-en-Provence, ...",1280a9ab-ebe0-4cdf-a340-74af2b3a02a2,2


In [15]:
# play with the color: continous/numerical
#px.set_mapbox_access_token(open(".mapbox_token").read())
import plotly.express as px
fig = px.scatter_mapbox(top_hotels_dtf, lat="gps_lat", lon="gps_long", color="score", text='hotel_name',
#                         size="hour",
                        color_continuous_scale=px.colors.cyclical.IceFire,                        
                        zoom=4.5,
                       width=900,
                        height=600,
                        labels={'score' : 'hotel score'},
                        title='Top hotels ' + weather_forecast_start_date.strftime("%d-%m-%Y") + ' to ' + weather_forecast_end_date.strftime("%d-%m-%Y"))
fig.update_traces(marker=dict(size=20))
fig.update_layout(mapbox_style="open-street-map")
fig.show()

In [14]:
fig.write_html("kayak_top20hotels_top5destinations.html")

### Export csvs to S3 (Storage pipeline) ###

In [22]:
from kayak.services import S3Service
import importlib

from kayak.pipelines import StoragePipeline,StoragePipelineOutput

In [23]:
session = boto3.Session()
S3_BUCKET_NAME = "pfe-kayak-bucket"
storage_pipeline_logger = root_logger.getChild("storage-pipeline")
s3_service = S3Service(S3_BUCKET_NAME,session,storage_pipeline_logger)
storage_pipeline = StoragePipeline(s3_service)

In [24]:
storage_output = storage_pipeline.upload_files(ingestion_outputs.to_file_list())

In [25]:
for s3_key in storage_output.uploaded_keys:
    print(s3_key)

csv/destination_gps_coordinates.csv
csv/weather/destination_weather_forecast_20260421T1200_20260426T0900.csv
csv/weather/destination_weather_scores_20260421T1200_20260426T0900.csv
csv/hotels/hotel_search_param_20260421T1200_20260426T0900.csv
csv/hotels/hotel_search_results_20260421T1200_20260426T0900.csv


### Load files stored on S3 with storage pipeline (LoadPipeline) ###

In [8]:
from kayak.pipelines import LoadPipeline
from kayak.services import S3Service

TEMP_DIR =  DATA_DIR / "temp"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

 

In [29]:
load_pipeline_logger = root_logger.getChild("load-pipeline")
load_pipeline = LoadPipeline(s3_service, TEMP_DIR,load_pipeline_logger)
file_path_list = load_pipeline.run(s3_keys=storage_output.uploaded_keys)

In [15]:

TEMP_DIR =  DATA_DIR / "temp"
TEMP_CSV_DIR = TEMP_DIR / "csv"
file_path_list = [
    TEMP_CSV_DIR / "destination_gps_coordinates.csv",
    TEMP_CSV_DIR / "weather" / "destination_weather_forecast_20260421T1200_20260426T0900.csv",
    TEMP_CSV_DIR / "weather" / "destination_weather_scores_20260421T1200_20260426T0900.csv",
    TEMP_CSV_DIR / "hotels" / "hotel_search_param_20260421T1200_20260426T0900.csv",
    TEMP_CSV_DIR / "hotels" / "hotel_search_results_20260421T1200_20260426T0900.csv",
]

for path in file_path_list:
    print(path, "->", path.exists())

C:\Pieric\datascience\JEDHA\Data Science Fundamentals\PYTHON\workspace\projects\jedha-dts-data-collect-mgmt-kayak\data\temp\csv\destination_gps_coordinates.csv -> True
C:\Pieric\datascience\JEDHA\Data Science Fundamentals\PYTHON\workspace\projects\jedha-dts-data-collect-mgmt-kayak\data\temp\csv\weather\destination_weather_forecast_20260421T1200_20260426T0900.csv -> True
C:\Pieric\datascience\JEDHA\Data Science Fundamentals\PYTHON\workspace\projects\jedha-dts-data-collect-mgmt-kayak\data\temp\csv\weather\destination_weather_scores_20260421T1200_20260426T0900.csv -> True
C:\Pieric\datascience\JEDHA\Data Science Fundamentals\PYTHON\workspace\projects\jedha-dts-data-collect-mgmt-kayak\data\temp\csv\hotels\hotel_search_param_20260421T1200_20260426T0900.csv -> True
C:\Pieric\datascience\JEDHA\Data Science Fundamentals\PYTHON\workspace\projects\jedha-dts-data-collect-mgmt-kayak\data\temp\csv\hotels\hotel_search_results_20260421T1200_20260426T0900.csv -> True


### Transform and load models to RDS Postgresql DB (DWH Pipeline) ###

In [16]:
import importlib
import kayak.pipelines

importlib.reload(kayak.pipelines)

<module 'kayak.pipelines' from 'C:\\Pieric\\datascience\\JEDHA\\Data Science Fundamentals\\PYTHON\\workspace\\projects\\jedha-dts-data-collect-mgmt-kayak\\src\\kayak\\pipelines\\__init__.py'>

In [17]:
from sqlalchemy import create_engine
from kayak.pipelines import DWHPipeline

# Make sure psycopg2-binary is installed on the kernel / anaconda environment pip install psycopg2-binary
rds_postgres_engine = create_engine(os.getenv("RDS_KAYAK_POSTGRESQL_URI"), echo=True)
dwh_pipeline_logger = root_logger.getChild("dwh-pipeline")

from kayak.models import Base

Base.metadata.create_all(rds_postgres_engine)

dwh_pipeline = DWHPipeline(rds_postgres_engine, dwh_pipeline_logger)
dwh_pipeline.run(file_path_list)

2026-04-21 12:54:19,518 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-21 12:54:19,518 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-21 12:54:19,618 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-21 12:54:19,619 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-21 12:54:19,717 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-21 12:54:19,718 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-21 12:54:19,818 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-21 12:54:19,823 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname